# Test Extraction Logic
In this notebook I wil test the various extractors to make sure they return the expected result

## Imports

In [1]:
import sys
import requests
import re
from bs4 import BeautifulSoup as bs
from copy import copy

from recipe_extractor.extraction.deterministic import DeterministicRecipeExtractor
from recipe_extractor.extraction.exceptions import UnsupportedSourceError
from recipe_extractor.data.schemas import RecipeData, IngredientData
from recipe_extractor.data.ingredients import normalize_ingredient, amount_convertor

## TwoPlaidAprons

In [31]:
def extract(url: str) -> RecipeData:
    ingredients = []

    page = requests.get(url)
    soup = bs(page.content)

    # Get name of the recipe, ingredients and preparation time and thumbnail
    title = soup.title.string[:-19]
    ings = soup.find_all('li', class_='wprm-recipe-ingredient')
    for ing in ings:
        ing_name = ing.find('span', class_='wprm-recipe-ingredient-name').get_text()
        ing_amount = ing.find('span', class_='wprm-recipe-ingredient-amount')
        if ing_amount:
            ing_amount = amount_convertor(ing_amount.get_text())
        ing_unit = ing.find('span', class_='wprm-recipe-ingredient-unit')
        if ing_unit:
            ing_unit = ing_unit.get_text()
        ing_norm_name = normalize_ingredient(ing_name)
        ingredients.append(
            IngredientData(
                name=ing_name,
                normalized_name=ing_norm_name,
                raw_text=str(ing),
                quantity=ing_amount,
                unit=ing_unit,
            )
        )

    time = soup.select('div.wprm-recipe-block-container.wprm-recipe-block-container-table.'+ 
        'wprm-block-text-normal.wprm-recipe-time-container.wprm-recipe-total-time-container '+
        'span.wprm-recipe-time.wprm-block-text-normal')
    thumb = soup.select('div.wprm-recipe-container div.wprm-recipe.wprm-recipe-template-template '+
        'div.wprm-container-float-right div.wprm-recipe-image.wprm-block-image-normal img')
    
    thumbnail = thumb[0].get('data-lazy-srcset').split(" ")[2]

    # Get total time in minutes
    if time:
        time = time[0].get_text()
    else:
        time = '0 minutes'
    
    matches = re.findall(r"(\d+)\s*(hour|hours|hr|minute|minutes|mins)", time, flags=re.IGNORECASE)
    
    total_minutes = 0
    for num, unit in matches:
        num = int(num)
        if unit.lower() in ["hour", "hr", "hours"]:
            total_minutes += num * 60
        else:
            total_minutes += num

    return title

In [32]:
ings = extract("https://twoplaidaprons.com/traditional-napa-cabbage-kimchi/")

In [33]:
ings

'Kimchi (Napa Cabbage Kimchi)'

In [12]:
print(ings[0])

<li class="wprm-recipe-ingredient" data-uid="0" style="list-style-type: none;"><span class="wprm-checkbox-container"><input aria-label=" 8 pounds napa cabbage (usually each head is around 4 pounds)" class="wprm-checkbox" id="wprm-checkbox-0" type="checkbox"/><label class="wprm-checkbox-label" for="wprm-checkbox-0"><span class="sr-only screen-reader-text wprm-screen-reader-text">▢ </span></label></span><span class="wprm-recipe-ingredient-amount">8</span> <span class="wprm-recipe-ingredient-unit">pounds</span> <span class="wprm-recipe-ingredient-name">napa cabbage</span> <span class="wprm-recipe-ingredient-notes wprm-recipe-ingredient-notes-smaller-faded">(usually each head is around 4 pounds)</span></li>


In [2]:
extractor = DeterministicRecipeExtractor()
recipe = extractor.extract(
    html = requests.get("https://twoplaidaprons.com/traditional-napa-cabbage-kimchi/").content,
    url = "https://twoplaidaprons.com/traditional-napa-cabbage-kimchi/"
)

In [3]:
print(recipe)

title='Kimchi (Napa Cabbage Kimchi)' url='https://twoplaidaprons.com/traditional-napa-cabbage-kimchi/' source='Two Plaid Aprons' total_time=135 thumbnail_url='https://twoplaidaprons.com/wp-content/uploads/2022/11/holding-kimchi-wrapped-up-thumbnail-360x360.jpg' ingredients=[IngredientData(name='napa cabbage', normalized_name='napa cabbage', raw_text='<li class="wprm-recipe-ingredient" data-uid="0" style="list-style-type: none;"><span class="wprm-checkbox-container"><input aria-label="\xa08 pounds napa cabbage (usually each head is around 4 pounds)" class="wprm-checkbox" id="wprm-checkbox-0" type="checkbox"/><label class="wprm-checkbox-label" for="wprm-checkbox-0"><span class="sr-only screen-reader-text wprm-screen-reader-text">▢ </span></label></span><span class="wprm-recipe-ingredient-amount">8</span> <span class="wprm-recipe-ingredient-unit">pounds</span> <span class="wprm-recipe-ingredient-name">napa cabbage</span> <span class="wprm-recipe-ingredient-notes wprm-recipe-ingredient-not

## Healthy Simple Yum

In [10]:
def extract_hsy(html, url):
    ingredients = []

    soup = bs(html, "html.parser")

    # Get name of the recipe
    title = soup.title.string

    # Get ingredients
    ings = soup.select('div.tasty-recipes-ingredients li')
    for ing in ings:
        raw_text = str(copy(ing))
        if ing.find("span", class_='nutrifox-quantity'):
            amount_span = ing.find("span", class_='nutrifox-quantity')
            ing_amount = amount_convertor(amount_span.get_text())
            amount_span.decompose()
            unit_span = ing.find("span", class_ = 'nutrifox-unit')
            ing_unit = unit_span.get_text()
            unit_span.decompose()
            ing_name = ing.get_text(strip=True)
            ing_norm_name = normalize_ingredient(ing_name)
        elif ing.find("span"):
            span = ing.find("span")
            ing_amount = span.get('data-amount')
            ing_unit = span.get('data-unit')
            span.decompose()
            ing_name = ing.get_text(strip=True)
            ing_norm_name = normalize_ingredient(ing_name)
        else: 
            ing_name = ing.get_text(strip=True)
            ing_norm_name = normalize_ingredient(ing_name)
            ing_amount = None
            ing_unit = None

        ingredients.append(
            IngredientData(
                name=ing_name,
                normalized_name=ing_norm_name,
                raw_text=raw_text,
                quantity=ing_amount,
                unit=ing_unit
            )
        )
        print(ing_norm_name)
        
    return ingredients

In [11]:
ings = extract_hsy(
    html = requests.get("https://healthysimpleyum.com/poblano-bean-soup/").content,
    url = "https://healthysimpleyum.com/poblano-bean-soup/"
)

vegetable broth
beans cooked
rice uncooked
poblano peppers roasted and diced
red bell pepper roasted and diced
onion diced
garlic cloves
oil
nutritional yeast
corn kernels
salsa verde
avocado diced
cilantro
tortilla chips


In [2]:
extractor = DeterministicRecipeExtractor()
recipe = extractor.extract(
    html = requests.get("https://healthysimpleyum.com/mexican-red-rice/").content,
    url = "https://healthysimpleyum.com/mexican-red-rice/"
)

In [3]:
print(recipe)

title='Mexican Red Rice (Authentic and Vegetarian)' url='https://healthysimpleyum.com/mexican-red-rice/' source='Healthy Simple Yum' total_time=None thumbnail_url='https://healthysimpleyum.com/wp-content/uploads/2025/12/mexican-red-rice-recipe-683x1024.jpg' ingredients=[IngredientData(name='jasmine rice', normalized_name='jasmine rice', raw_text='<li><span class="nutrifox-quantity" data-amount="2" data-nf-food-description="Rice, white, medium-grain, cooked, unenriched" data-nf-food-id="6618" data-nf-metric="372" data-nf-metric-unit="gram" data-nf-original="usc" data-nf-usc="2" data-nf-usc-unit="cup" data-unit="cup">2</span> <span class="nutrifox-unit" data-nf-food-description="Rice, white, medium-grain, cooked, unenriched" data-nf-food-id="6618" data-nf-metric="gram" data-nf-original="usc" data-nf-usc="cup">cups</span> jasmine rice</li>', quantity=2.0, unit='cups'), IngredientData(name='warm water', normalized_name='water', raw_text='<li><span class="nutrifox-quantity" data-amount="2" 